1.
<p>Generate 400 used two-wheeler listings for a fictional Gujarat marketplace in a notebook named 'bike_cleaning.ipynb' with seed 42: listing_id, brand (Veloce, Rajhans, Sundar, Tarang), city (Ahmedabad, Surat, Vadodara, Rajkot), model_year 2008 to 2023, km_driven 2000 to 90000, owner_count 1 to 3, price_inr 25000 to 140000. Blank 8% of km_driven at random, half of owner_count where model_year is below 2013, and the top 5% of price_inr. Save 'bike_listings_raw.csv'. Hint: Use `quantile(0.95)` for the price cut-off.</p>
2.
<p>Build `missing_report` for the raw listings with the count and percentage of missing values per column using `isna()`, the number of fully complete rows using `notnull()`, and a `mechanism` column (MCAR for km_driven, MAR for owner_count, MNAR for price_inr). Add the owner_count missing rate for model_year below 2013 against 2013 and later. Constraint: Do not use any auto-EDA library.</p>
3.
<p>Create three frames from the raw listings: `drop_any` with `dropna(how='any')`, `drop_thresh` with `dropna(thresh=6)` and `drop_cols` with `dropna(axis=1, thresh=360)`. Print a table of row count, column count and mean price_inr for the raw data, `drop_any` and `drop_thresh`. Hint: `thresh` is the minimum number of non-missing values needed to keep a row or column.</p>
4.
<p>Create `bikes_filled`: km_driven with the median, owner_count with `mode()[0]`, and price_inr with `ffill()` after sorting by model_year, then `bfill()` for leftovers. Print mean price_inr for the raw non-missing rows, `drop_any` and `bikes_filled`, confirm zero missing values, and save 'bike_listings_s6.csv'. Run Restart & Run All and push to 's06_bike_cleaning' in your GitHub repo 'ds_analysis_portfolio'.</p>

In [1]:
import numpy as np
import pandas as pd

In [2]:
""" Task 1 
"""

np.random.seed(42)

bikes = pd.DataFrame({
    "listing_id": range(1, 401),

    "brand": np.random.choice(
        ["Veloce", "Rajhans", "Sundar", "Tarang"], 400
    ),

    "city": np.random.choice(
        ["Ahmedabad", "Surat", "Vadodara", "Rajkot"], 400
    ),

    "model_year": np.random.randint(2008, 2024, 400),

    "km_driven": np.random.randint(2000, 90001, 400),

    "owner_count": np.random.randint(1, 4, 400),

    "price_inr": np.random.randint(25000, 140001, 400)
})

print(bikes.head())
print("Shape:", bikes.shape)

   listing_id   brand       city  model_year  km_driven  owner_count  \
0           1  Sundar     Rajkot        2016      89044            3   
1           2  Tarang     Rajkot        2021      89505            2   
2           3  Veloce   Vadodara        2008      38668            3   
3           4  Sundar  Ahmedabad        2008      25061            1   
4           5  Sundar     Rajkot        2021      81252            3   

   price_inr  
0      51292  
1     136785  
2     111206  
3      94274  
4      27469  
Shape: (400, 7)


In [3]:

raw = bikes.copy()

missing_km = np.random.choice(
    raw.index,
    size=32,
    replace=False
)

raw.loc[missing_km, "km_driven"] = np.nan


old_bikes = raw[raw["model_year"] < 2013].index

missing_owner = np.random.choice(
    old_bikes,
    size=len(old_bikes) // 2,
    replace=False
)

raw.loc[missing_owner, "owner_count"] = np.nan


price_cutoff = raw["price_inr"].quantile(0.95)

raw.loc[
    raw["price_inr"] >= price_cutoff,
    "price_inr"
] = np.nan


raw.to_csv("bike_listings_raw.csv", index=False)

print("Raw bike CSV created!")
print("Price cutoff:", price_cutoff)
print(raw.head())

Raw bike CSV created!
Price cutoff: 136573.15
   listing_id   brand       city  model_year  km_driven  owner_count  \
0           1  Sundar     Rajkot        2016    89044.0          3.0   
1           2  Tarang     Rajkot        2021    89505.0          2.0   
2           3  Veloce   Vadodara        2008    38668.0          NaN   
3           4  Sundar  Ahmedabad        2008        NaN          1.0   
4           5  Sundar     Rajkot        2021    81252.0          3.0   

   price_inr  
0    51292.0  
1        NaN  
2   111206.0  
3    94274.0  
4    27469.0  


In [ ]:
""" Task 2
"""
missing_count = raw.isna().sum()
missing_percentage = (raw.isna().mean() * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})


missing_report["mechanism"] = [
    "None",
    "None",
    "None",
    "None",
    "MCAR",
    "MAR",
    "MNAR"
]

print("Missing Report:")
print(missing_report)


complete_rows = raw.notnull().all(axis=1).sum()

print("Fully Complete Rows:", complete_rows)

Missing Report:
             missing_count  missing_percentage mechanism
listing_id               0                0.00      None
brand                    0                0.00      None
city                     0                0.00      None
model_year               0                0.00      None
km_driven               32                8.00      MCAR
owner_count             67               16.75       MAR
price_inr               20                5.00      MNAR
Fully Complete Rows: 289


In [ ]:

before_2013 = raw[raw["model_year"] < 2013]

from_2013 = raw[raw["model_year"] >= 2013]


old_missing_rate = before_2013["owner_count"].isna().mean() * 100

new_missing_rate = from_2013["owner_count"].isna().mean() * 100

print("Owner missing rate below 2013:", round(old_missing_rate, 2), "%")

print("Owner missing rate 2013 and later:", round(new_missing_rate, 2), "%")

Owner missing rate below 2013: 49.63 %
Owner missing rate 2013 and later: 0.0 %


In [ ]:
""" Task 3
"""

drop_any = raw.dropna(how="any")


drop_thresh = raw.dropna(thresh=6)

drop_cols = raw.dropna(axis=1, thresh=360)

print("Drop Any Shape:", drop_any.shape)
print("Drop Threshold Shape:", drop_thresh.shape)
print("Drop Columns Shape:", drop_cols.shape)

Drop Any Shape: (289, 7)
Drop Threshold Shape: (392, 7)
Drop Columns Shape: (400, 6)


In [7]:
comparison = pd.DataFrame({
    "Dataset": ["Raw Data", "Drop Any", "Drop Threshold"],

    "Row Count": [
        len(raw),
        len(drop_any),
        len(drop_thresh)
    ],

    "Column Count": [
        len(raw.columns),
        len(drop_any.columns),
        len(drop_thresh.columns)
    ],

    "Mean Price": [
        raw["price_inr"].mean(),
        drop_any["price_inr"].mean(),
        drop_thresh["price_inr"].mean()
    ]
})

print(comparison)

          Dataset  Row Count  Column Count    Mean Price
0        Raw Data        400             7  79096.457895
1        Drop Any        289             7  80949.539792
2  Drop Threshold        392             7  79090.135638


In [8]:
""" Task 4
"""
# Create a copy of raw data
bikes_filled = raw.copy()

# Fill km_driven with median
bikes_filled["km_driven"] = bikes_filled["km_driven"].fillna(
    bikes_filled["km_driven"].median()
)

# Fill owner_count with mode
bikes_filled["owner_count"] = bikes_filled["owner_count"].fillna(
    bikes_filled["owner_count"].mode()[0]
)

# Sort by model year
bikes_filled = bikes_filled.sort_values("model_year")

# Fill price using forward fill
bikes_filled["price_inr"] = bikes_filled["price_inr"].ffill()

# Fill remaining values using backward fill
bikes_filled["price_inr"] = bikes_filled["price_inr"].bfill()

print(bikes_filled.head())

     listing_id    brand      city  model_year  km_driven  owner_count  \
357         358   Tarang  Vadodara        2008    27522.0          2.0   
355         356   Sundar     Surat        2008    13485.0          3.0   
349         350   Sundar     Surat        2008    40258.0          3.0   
341         342   Tarang     Surat        2008    61676.0          1.0   
65           66  Rajhans  Vadodara        2008    28069.0          1.0   

     price_inr  
357    51958.0  
355   102539.0  
349    90128.0  
341    50963.0  
65     50963.0  


In [9]:
# Mean price comparisons
raw_mean = raw["price_inr"].mean()

drop_mean = drop_any["price_inr"].mean()

filled_mean = bikes_filled["price_inr"].mean()

print("Raw non-missing mean price:", raw_mean)

print("Drop Any mean price:", drop_mean)

print("Filled mean price:", filled_mean)

# Check missing values
print("\nMissing values after filling:")
print(bikes_filled.isna().sum())

print("\nTotal missing values:", bikes_filled.isna().sum().sum())

Raw non-missing mean price: 79096.45789473684
Drop Any mean price: 80949.53979238754
Filled mean price: 78436.8725

Missing values after filling:
listing_id     0
brand          0
city           0
model_year     0
km_driven      0
owner_count    0
price_inr      0
dtype: int64

Total missing values: 0


In [10]:
bikes_filled.to_csv("bike_listings_s6.csv", index=False)

print("Cleaned bike data saved successfully!")
print("Final shape:", bikes_filled.shape)

Cleaned bike data saved successfully!
Final shape: (400, 7)
